# Từ đồ thị đến embedding trên Moons

Mini-lab của D dùng 40 điểm với seed 42. Từ dữ liệu đã chuẩn hóa, ta dựng đồ thị RBF, tính Laplacian chưa chuẩn hóa \(L=D-W\), rồi chạy K-Means trên embedding từ hai vector riêng.

Biến thể chính thức của nhóm sẽ dùng \(L_{\mathrm{sym}}\) và chuẩn hóa từng hàng của embedding sau khi B bàn giao pipeline.

In [1]:
import sys

print("Python đang dùng:", sys.executable)

%pip install -e "D:/Kiet Tin/Tin_DH/CS115/Spectral-clustering"

Python đang dùng: d:\Kiet Tin\.venv\Scripts\python.exe
Obtaining file:///D:/Kiet%20Tin/Tin_DH/CS115/Spectral-clustering
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl (12.6 MB)
  Building editable for spectral-clustering-project (pyproject.toml): started
  Building editable for spectral-clustering-project (pyproject.toml): finished with status 'done'
  Created wheel for spectral-clustering-project: filename=spectral_clusterin


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from spectral_clustering.baselines import run_kmeans
from spectral_clustering.datasets import generate_moons
from spectral_clustering.graphs import (
    degree_matrix,
    rbf_affinity,
    unnormalized_laplacian,
)
from spectral_clustering.preprocessing import standardize_features
from spectral_clustering.visualization import plot_clusters

n_samples = 40
seed = 42
gamma = 1.0

X, y = generate_moons(n_samples=n_samples, seed=seed)
X_scaled = standardize_features(X)
W = rbf_affinity(X_scaled, gamma=gamma)
D = degree_matrix(W)
L = unnormalized_laplacian(W)

assert X_scaled.shape == (n_samples, 2)
assert W.shape == D.shape == L.shape == (n_samples, n_samples)
np.testing.assert_allclose(L.sum(axis=1), 0.0, atol=1e-12)

ModuleNotFoundError: No module named 'spectral_clustering'

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4), layout="constrained")

for ax, matrix, name in zip(axes, (W, D, L), ("W", "D", "L")):
    image = ax.imshow(matrix, cmap="coolwarm", interpolation="nearest")
    ax.set_title(name)
    ax.set_xlabel("Chỉ số cột")
    ax.set_ylabel("Chỉ số hàng")
    fig.colorbar(image, ax=ax, fraction=0.05)

plt.show()

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(L)
U = eigenvectors[:, :2]
spectral_labels = run_kmeans(U, n_clusters=2, random_state=seed)

assert U.shape == (n_samples, 2)
assert spectral_labels.shape == (n_samples,)
print("Sáu trị riêng nhỏ nhất:", np.round(eigenvalues[:6], 6))

fig, ax = plt.subplots(figsize=(5, 3), layout="constrained")
ax.plot(np.arange(1, 7), eigenvalues[:6], marker="o")
ax.set_title("Sáu trị riêng nhỏ nhất của L")
ax.set_xlabel("Thứ tự trị riêng")
ax.set_ylabel("Giá trị")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")

plot_clusters(X_scaled, y, ax=axes[0], title="Ground Truth trên X đã chuẩn hóa")
plot_clusters(U, spectral_labels, ax=axes[1], title="Nhãn dự đoán trên embedding U")
plot_clusters(X_scaled, spectral_labels, ax=axes[2], title="Nhãn dự đoán trên X đã chuẩn hóa")

axes[1].set_aspect("auto")
axes[1].set_xlim(U[:, 0].min() - 0.05, U[:, 0].max() + 0.05)
axes[1].set_xlabel("Vector riêng 1")
axes[1].set_ylabel("Vector riêng 2")
plt.show()

Mỗi hàng của \(U\) tương ứng với một dòng của \(X_{\mathrm{scaled}}\). Với đồ thị RBF liên thông ở đây, vector riêng đầu tiên của Laplacian chưa chuẩn hóa gần như hằng, nên embedding hiện thành một dải dọc. Hai hình dự đoán dùng cùng một mảng nhãn để có thể đối chiếu kết quả giữa hai không gian. Màu ở hình Ground Truth thể hiện mã nhãn thật. Mã nhãn giữa hai phân hoạch có thể hoán đổi.